In [1]:
import pandas as pd


In [2]:
orders = pd.read_csv('../data/processed/orders_clean.csv', 
                     parse_dates=['order_purchase_timestamp',
                                  'order_approved_at',
                                  'order_delivered_carrier_date',
                                  'order_delivered_customer_date',
                                  'order_estimated_delivery_date'])

order_items = pd.read_csv('../data/processed/order_items_clean.csv',
                          parse_dates=['shipping_limit_date'])

customers = pd.read_csv('../data/processed/customers_clean.csv')
products = pd.read_csv('../data/processed/products_clean.csv')
sellers = pd.read_csv('../data/processed/sellers_clean.csv')
payments = pd.read_csv('../data/processed/payments_clean.csv')
reviews = pd.read_csv('../data/processed/reviews_clean.csv')
geolocation = pd.read_csv('../data/processed/geolocation_clean.csv')
translation = pd.read_csv('../data/processed/translation_clean.csv')

print("All datasets loaded from processed folder")

All datasets loaded from processed folder


In [3]:
df = order_items.copy()
print("Starting shape", df.shape)
display("Starting columns", df.columns.tolist())

Starting shape (112650, 7)


'Starting columns'

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [4]:
df = df.merge(orders[['order_id', 
                        'customer_id',
                        'order_status',
                        'order_purchase_timestamp']], 
              on='order_id', 
              how='left')

print("After joining orders:", df.shape)

After joining orders: (112650, 10)


In [5]:
df = df[df['order_status'] == 'delivered']
print("After filtering to delivered only:", df.shape)
print("Unique statuses remaining:", df['order_status'].unique())

After filtering to delivered only: (110197, 10)
Unique statuses remaining: <StringArray>
['delivered']
Length: 1, dtype: str


In [6]:
df = df.merge(customers[['customer_id', 
                           'customer_state']], 
              on='customer_id', 
              how='left')

print("After joining customers:", df.shape)
print("Sample states:", df['customer_state'].unique()[:5])

After joining customers: (110197, 11)
Sample states: <StringArray>
['RJ', 'SP', 'MG', 'PR', 'GO']
Length: 5, dtype: str


In [7]:
df['total_revenue'] = df['price'] + df['freight_value']
print("Revenue column created")
print(df['total_revenue'].describe())

Revenue column created
count    110197.000000
mean        139.929161
std         189.319151
min           6.080000
25%          55.180000
50%          92.130000
75%         157.510000
max        6929.310000
Name: total_revenue, dtype: float64


In [8]:
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
display("Sample months:", df['order_month'].unique()[:5])

#print(orders['order_purchase_timestamp'].dtype)

'Sample months:'

<PeriodArray>
['2017-09', '2017-04', '2018-01', '2018-08', '2017-02']
Length: 5, dtype: period[M]

In [11]:
# Revenue by month
revenue_by_month = df.groupby('order_month')['total_revenue'].sum().reset_index()
revenue_by_month.columns = ['month', 'total_revenue']
revenue_by_month = revenue_by_month.sort_values('month').head(10)
display(revenue_by_month)

,month,total_revenue
0,2016-09,143.46
1,2016-10,46490.66
2,2016-12,19.62
3,2017-01,127482.37
4,2017-02,271239.32
5,2017-03,414330.95
6,2017-04,390812.40
7,2017-05,566851.40
8,2017-06,490050.37
9,2017-07,566299.08


In [10]:
# Revenue by state
revenue_by_state = df.groupby('customer_state')['total_revenue'].sum().reset_index()
revenue_by_state.columns = ['state', 'total_revenue']
revenue_by_state = revenue_by_state.sort_values('total_revenue', ascending=False).head(10)
display(revenue_by_state)

,state,total_revenue
25,SP,5769703.15
18,RJ,2055401.57
10,MG,1818891.67
22,RS,861472.79
17,PR,781708.80
23,SC,595127.78
4,BA,591137.81
6,DF,346123.35
8,GO,334212.35
7,ES,317657.93
